# Interactive Inference Pipeline

This notebook breaks down the `inference_pipeline.py` script into interactive cells, allowing you to manually inspect each task step-by-step.

## 1. Imports and Setup

In [ ]:
import torch
import requests
import numpy as np
import pandas as pd
from PIL import Image, ImageDraw
from io import BytesIO
import os
import json
import gc
from tqdm.notebook import tqdm
import cv2
import matplotlib.pyplot as plt
import random

from transformers import AutoProcessor, AutoModelForCausalLM 
from peft import LoraConfig, get_peft_model, PeftModel

from huggingface_hub import login
login(token="api_here")

torch_dtype = torch.float32

# Please replace paths with your actual model paths
try:
    model = AutoModelForCausalLM.from_pretrained("your/path/to/model", torch_dtype=torch_dtype, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained("your/path/to/model", trust_remote_code=True)
except Exception as e:
    print("Saved model not found initialzing new")
    model = AutoModelForCausalLM.from_pretrained("microsoft/Florence-2-base", torch_dtype=torch_dtype, trust_remote_code=True)
    processor = AutoProcessor.from_pretrained("microsoft/Florence-2-base", trust_remote_code=True)

model = PeftModel.from_pretrained(
    model,
    "peeache/Florence-2-Final-Batched-PipeLine-LoRA-128-256",
    torch_dtype=torch_dtype,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"Model loaded on {device}")

## 2. Helper Functions

In [ ]:
import re
import torch.nn.functional as F
from nltk.corpus import stopwords

# Preload stopwords (English)
STOPWORDS = set(stopwords.words("english"))
STOPWORDS.add("image")
STOPWORDS.add("view")

colormap = ['blue','orange','green','purple','brown','pink','gray','olive','cyan','red',
            'lime','indigo','violet','aqua','magenta','coral','gold','tan','skyblue']

def draw_polygons(image, prediction, fill_mask=True):
    draw = ImageDraw.Draw(image)
    scale = 1
    for polygons, label in zip(prediction['polygons'], prediction['labels']):    
        color = random.choice(colormap)
        fill_color = random.choice(colormap) if fill_mask else None
        for _polygon in polygons:
            _polygon = np.array(_polygon).reshape(-1, 2)
            if len(_polygon) < 3:
                continue
            _polygon = (_polygon * scale).reshape(-1).tolist()
            if fill_mask:
                draw.polygon(_polygon, outline=color, fill=fill_color)
            else:
                draw.polygon(_polygon, outline=color)
            draw.text((_polygon[0] + 8, _polygon[1] + 2), label, fill=color)

    plt.imshow(np.array(image))
    plt.axis('off')
    plt.title("Model Output Overlay")
    plt.show()
    return image

def create_mask(image, prediction):
    mask = Image.new('L', image.size, 0)  
    draw = ImageDraw.Draw(mask)
    scale = 1
    for polygons, label in zip(prediction['polygons'], prediction['labels']): 
        for _polygon in polygons:
            _polygon = np.array(_polygon).reshape(-1, 2)
            if len(_polygon) < 3:
                continue
            _polygon = (_polygon * scale).reshape(-1).tolist()
            draw.polygon(_polygon, outline=255, fill=255)

    plt.imshow(np.array(mask), cmap='gray')
    plt.axis('off')
    plt.title("Black and White Mask / Visualization")
    plt.show()
    return np.array(mask, dtype=np.uint8)

def apply_mask_and_save(image, mask, output_path, alpha=0.3, mask_color=(255, 0, 0)):
    mask_bool = (mask > 128).astype(np.uint8)
    masked_image = image.copy()
    overlay = np.full_like(image, mask_color, dtype=np.uint8)
    blend = cv2.addWeighted(image, 1 - alpha, overlay, alpha, 0)
    masked_image[mask_bool == 1] = blend[mask_bool == 1]
    cv2.imwrite(output_path, masked_image)

def explain_with_token_probs_and_conf(model, processor, image, question, device, 
                                      max_new_tokens=128, k=5):
    prompt = f"<MedVQA_EXPLAIN> {question} Explain in Detail."
    inputs = processor(text=prompt, images=[image],
                       return_tensors="pt", padding=True)
    inputs = {k: v.to(device) for k, v in inputs.items()}

    decoder_input_ids = torch.tensor(
        [[model.generation_config.decoder_start_token_id]],
        device=device
    )

    all_token_info = []
    stability_scores = []  

    for step in range(max_new_tokens):
        with torch.no_grad():
            outputs = model(
                input_ids=inputs["input_ids"],        
                pixel_values=inputs["pixel_values"],  
                decoder_input_ids=decoder_input_ids   
            )
            logits = outputs.logits[:, -1, :]  
            probs = F.softmax(logits, dim=-1)

            topk_probs, _ = torch.topk(probs, k, dim=-1)
            topk_mass = topk_probs.sum().item()
            stability_scores.append(topk_mass)

            next_id = torch.argmax(probs, dim=-1)
            next_prob = probs[0, next_id].item()

        token_str = processor.tokenizer.decode([next_id.item()], skip_special_tokens=True)
        all_token_info.append({
            "token_id": next_id.item(),
            "token_str": token_str,
            "prob": next_prob,
            "topk_mass": topk_mass
        })

        decoder_input_ids = torch.cat([decoder_input_ids, next_id.unsqueeze(0)], dim=-1)
        if next_id.item() == processor.tokenizer.eos_token_id:
            break

    generated_text = processor.batch_decode(decoder_input_ids, skip_special_tokens=False)[0]
    vqa_answer_exp = processor.post_process_generation(
        generated_text,
        task="<MedVQA_EXP>",
        image_size=(image.width, image.height)
    )['<MedVQA_EXP>']

    # ---- Final Robust Confidence ----
    meaningful_masses = []
    for info in all_token_info:
        word = info["token_str"].strip().lower()
        if word and word not in STOPWORDS and not re.match(r"^<.*>$", word):
            meaningful_masses.append(info["topk_mass"])
            
    if not meaningful_masses:
        meaningful_masses = stability_scores

    if meaningful_masses:
        avg_mass = float(np.mean(meaningful_masses))
        p10_mass = float(np.percentile(meaningful_masses, 10))
        topk_stability_conf = (0.5 * avg_mass) + (0.5 * p10_mass)
    else:
        topk_stability_conf = 0.0

    return vqa_answer_exp, all_token_info, topk_stability_conf

## 3. Configuration & Test Instance Setup
Set `USE_DATASET = True` to randomly pick from the Kvasir test set.
Or set `USE_DATASET = False` to provide an image id and a question manually.

In [ ]:
USE_DATASET = True

IMG_DIR = "data/images"  # Directory containing images

if USE_DATASET:
    from datasets import load_dataset, Image as HfImage
    ds = load_dataset("SimulaMet/Kvasir-VQA-x1")["test"]
    #btw here complexitiy = 1 questions are getting filtered (you can remove this filter to get any type of complexitiy but
    #note that the explaination and segmentation works on the answers of complexitiy = 1 questions)
    val_set_task2 = (
        ds.filter(lambda x: x["complexity"] == 1)
          .shuffle(seed=42)
          .select(range(min(1500, len(ds))))
    )
    
    # Select the sample instance
    sample_idx = 0
    img_id = val_set_task2['img_id'][sample_idx]
    question = val_set_task2['question'][sample_idx]
    print(f"[Mode: Dataset] Loaded index {sample_idx}.\nImgID: {img_id}\nQuestion: {question}")

else:
    # Fill these in manually:
    img_id = "cl8k2u1pm1dw7083203g1b7yv"  # Replace with actual ID string
    question = "Is there any abnormility in the image?"
    print(f"[Mode: Manual]\nImgID: {img_id}\nQuestion: {question}")

image_path = f"{IMG_DIR}/{img_id}.jpg"

# Load and verify image existence
if os.path.exists(image_path):
    image = cv2.imread(image_path)
    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    # Convert to PIL for the processor
    pil_image = Image.fromarray(image)

    plt.figure(figsize=(6,6))
    plt.imshow(pil_image)
    plt.axis('off')
    plt.title(f"Input Image: {img_id}")
    plt.show()
else:
    print(f"Warning: Image not found at path {image_path}. Please correct the string above.")
    pil_image = None

## 4. Task 1: Basic VQA Inference (`<MedVQA>`)

In [1]:
prompt = f"<MedVQA> {question}"
inputs = processor(text=prompt, images=[pil_image],
                   return_tensors="pt", padding=True)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
vqa_answer = processor.post_process_generation(
    generated_text, task="<MedVQA>",
    image_size=(pil_image.width, pil_image.height)
)['<MedVQA>']

print("✅ VQA Answer:", vqa_answer)

NameError: name 'question' is not defined

## 5. Task 2: Explainability Generation & Confidence Score

In [ ]:
vqa_answer_exp, token_probs, topk_stability_conf = explain_with_token_probs_and_conf(
    model, processor, pil_image, question, device
)

print("Detailed Explanation:\n", vqa_answer_exp)
print(f"\n Calculated Confidence Score: {topk_stability_conf:.4f}")

## 6. Task 3: Referring Expression Segmentation
Locates the regions corresponding to the given VQA answer.

In [ ]:
inputs = processor(
    text=f"<REFERRING_EXPRESSION_SEGMENTATION> {vqa_answer}",
    images=[pil_image],
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
seg_answer = processor.post_process_generation(
    generated_text,
    task="<REFERRING_EXPRESSION_SEGMENTATION>",
    image_size=(pil_image.width, pil_image.height)
)['<REFERRING_EXPRESSION_SEGMENTATION>']

print(" Localized Labels:", seg_answer.get('labels', []))

# Create visualizations around them:
drawn_image = draw_polygons(pil_image.copy(), seg_answer)
mask = create_mask(pil_image, seg_answer)

## 7. Task 4: Detailed Scene Captioning

In [ ]:
inputs = processor(
    text=f"<MORE_DETAILED_CAPTION>",
    images=[pil_image],
    return_tensors="pt",
    padding=True
)
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
    )

generated_text = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
caption_answer = processor.post_process_generation(
    generated_text,
    task="<MORE_DETAILED_CAPTION>",
    image_size=(pil_image.width, pil_image.height)
)['<MORE_DETAILED_CAPTION>']

print("Image Caption:\n", caption_answer)

## 8. Export Visuals & Aggregate JSON Output

In [ ]:
vis_dir = "visuals"
os.makedirs(vis_dir, exist_ok=True) 

# Composite masked layer
vis_path = f"{vis_dir}/_mask_{img_id}_interactive.jpg"
image_bgr = cv2.cvtColor(np.array(pil_image), cv2.COLOR_RGB2BGR)
apply_mask_and_save(image_bgr, mask, vis_path)
print(f"Mask validation image saved to: {vis_path}")

compiled_record = {
    "img_id": img_id,
    "question": question,
    "answer": vqa_answer,
    "textual_explanation": f"{vqa_answer_exp}\nOverall explanation of image: {caption_answer}",
    "visual_explanation": [{
        "type": "segmentation_mask",
        "data": vis_path,
        "description": "Highlighted mask showing the region of interest supporting the answer."
    }],
    "confidence_score": float(topk_stability_conf)
}

print("\n=== FINAL SUMMARY RECORD ===")
print(json.dumps(compiled_record, indent=2))